This version deals with five datasets: crc01 (scRNA-seq, HRA003293), crc03 (scRNA-seq, GSE234804), crc06 (snRNA-seq, GSE183916), crc09 (Pelka 2021), crc10 (Lee 2020)

In [1]:
### Import modules
import os
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import scipy as sp
import anndata as ad
import sys
import re
### Import personal Utensils
sys.path.insert(0, "..")
import utils as ut

### Set Data Dir

In [2]:
DataRootDir = Path("../../data")

### Data Curation

Doctrine:

A. raw data will be stored in .layers\['counts']

B. Unification of .obs metadata
- Glob_CellId (used as index): study name + sample specific barcode
- Glob_Sample: study name + sample
- Glob_Patient: study name + patient
- Patient: nomification of the original study
- Study: study name
- Tissue: crc or pm or normal
- CellTypeFull: store layered annotation
- CellType: store final annotation
- Batch: Batch information per study

C. Unification of .var metadata
- var_names or index will be ENSG ID for following analysis
- gene_id: replicate of var_names
- gene_name: gene symbol

#### Step one: read in and unify data type and meta data(obs)

In [3]:
### ======
# crc01 data
### ======
CurrentDir = DataRootDir / "crc01"
adata_crc01 = sc.read_h5ad(CurrentDir / "HRA003293.h5ad")

# remove log data and transform raw counts to int32
del adata_crc01.layers["data"]
del adata_crc01.X
adata_crc01.layers["counts"] = ut.to_int32_counts(
    adata_crc01.layers["counts"]
)

In [4]:
# modify obs
crc01_obs = adata_crc01.obs.copy()
obs_list = ['sample','cells','tissue','patient','method','celltype']
crc01_obs = crc01_obs[obs_list]
crc01_obs['Study'] = 'crc01'
rename_dict = {"patient": "Patient","tissue":"Tissue","sample":"Sample","cells":"CellID", "method":"Batch","celltype":"CellType"}
crc01_obs = crc01_obs.rename(columns=rename_dict)
crc01_obs['Glob_CellID'] = crc01_obs['Study'] + '-' + crc01_obs['CellID']
crc01_obs['Glob_Sample'] = crc01_obs['Study'] + '-' + crc01_obs['Sample']
crc01_obs['Glob_Patient'] = crc01_obs['Study'] + '-' + crc01_obs['Patient']
crc01_obs['Batch'] = 'scRNA_' + crc01_obs['Batch']
crc01_obs = crc01_obs.set_index("Glob_CellID", drop=False)
adata_crc01.obs = crc01_obs.copy()
del crc01_obs

In [5]:
# rule out liver metastasis
adata_crc01 = adata_crc01[adata_crc01.obs["Tissue"] != "lm"].copy()

In [6]:
# reset other analysis
adata_crc01.obsm = {}
adata_crc01.obsp = {}
# reset var
adata_crc01.var = pd.DataFrame({'gene_name': adata_crc01.var_names})

/home/zhekai/miniconda3/envs/pm/lib/python3.13/functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [7]:
### ======
# crc03 data (scRNA-seq, GSE234804)
### ======
CurrentDir = DataRootDir / "crc03/GSE234804"
FileList = sorted(
    file for file in os.listdir(CurrentDir) if file.endswith(".h5ad")
)
adata_list = []

for file in FileList:
    name = Path(file).stem.split("_")[1]
    adata = sc.read_h5ad(CurrentDir / file)

    # Clean up layers
    for layer_name in list(adata.layers.keys()):
        if layer_name == "scale.data" or layer_name == "data" :
            del adata.layers[layer_name]
            continue
        elif layer_name == "counts":
            adata.layers[layer_name] = sp.sparse.csr_matrix(
                ut.to_int32_counts(adata.layers[layer_name])
            )
            continue
    
    # Build metadata
    adata.obs["Study"] = "crc03"
    adata.obs["Sample"] = name
    adata.obs["Batch"] = "scRNA"
    adata.obs["Glob_CellID"] = "crc03-" + name + "_" + adata.obs.index
    adata.obs["Glob_Sample"] = "crc03-" + name

    # use regex to find prefix and num
    prefix, num = re.findall(r"^(\w+)(\d+)$", name)[0]
    if prefix == "PC":
        tissue = "pm"
    elif prefix == "CRC":
        tissue = "crc"
    patient = "patient-" + num
    
    # assign value 
    adata.obs["Tissue"] = tissue
    adata.obs["Patient"] = patient
    adata.obs["Glob_Patient"] = "crc03-" + adata.obs["Patient"]
    adata.obs = adata.obs.set_index("Glob_CellID", drop=False)

    adata_list.append(adata)

In [8]:
adata_crc03 = ad.concat(
    adata_list,
    join="outer",
    merge="first",
    uns_merge=None,
)
del adata_list
# Reset analysis artifacts, keep var (contains gene_ids)
adata_crc03.obsm = {}
adata_crc03.obsp = {}

In [9]:
### ======
# crc06 data (snRNA-seq, GSE183916)
### ======
adata_crc06 = sc.read_h5ad(DataRootDir / "crc06/GSE183916/GSE183916.h5ad")
# only use raw data
del adata_crc06.layers["data"]
adata_crc06.layers["counts"] = ut.to_int32_counts(adata_crc06.layers["counts"])

In [10]:
# adjusting obs
crc06_obs = adata_crc06.obs.copy()
obs_list = ['loc','pat_id']
crc06_obs = crc06_obs[obs_list]
crc06_obs["Study"] = "crc06"
rename_dict = {"pat_id": "Patient","loc": "Tissue",}
crc06_obs = crc06_obs.rename(columns=rename_dict)
crc06_obs["Tissue"] = crc06_obs["Tissue"].map({"metastasis": "pm","primary": "crc",})
crc06_obs["Sample"] = crc06_obs["Patient"] + "_" + crc06_obs["Tissue"]
crc06_obs["Batch"] = "snRNA"
crc06_obs["Glob_CellID"] = ("crc06-" + crc06_obs.index)
crc06_obs["Glob_Sample"] = "crc06-" + crc06_obs["Sample"]
crc06_obs["Glob_Patient"] = "crc06-" + crc06_obs["Patient"]
# dealing index
crc06_obs['CellID'] = crc06_obs.index
crc06_obs = crc06_obs.set_index("Glob_CellID", drop=False)
adata_crc06.obs = crc06_obs.copy()
del crc06_obs
adata_crc06.var = pd.DataFrame({'gene_name':adata_crc06.var.index})

/home/zhekai/miniconda3/envs/pm/lib/python3.13/functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [11]:
### ======
# crc09 data (scRNA)
### ======
# getting raw data
# here, .X stores integer
adata_crc09 = sc.read_h5ad(DataRootDir / "crc09/pelka-2021.h5ad")
adata_crc09.layers['counts']= ut.to_int32_counts(adata_crc09.X.copy())
del adata_crc09.X

/home/zhekai/miniconda3/envs/pm/lib/python3.13/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [12]:
# getting info from obs
crc09_obs = adata_crc09.obs.copy()
obs_list = ['batchID','clTopLevel','clMidwayPr','cl295v11SubFull','SPECIMEN_TYPE','PROCESSING_TYPE','SINGLECELL_TYPE','PID']
# transform categorical to string
for name in obs_list:
    crc09_obs[name] = crc09_obs[name].astype(str)
crc09_obs = crc09_obs[obs_list]
crc09_obs['CellID'] = crc09_obs.index.astype(str)
rename_dict = {'PID':'Patient', 'batchID':'Sample','SPECIMEN_TYPE':'Tissue','SINGLECELL_TYPE':'Batch','clMidwayPr':'CellType'}
crc09_obs = crc09_obs.rename(columns = rename_dict)
crc09_obs['Tissue'] = crc09_obs['Tissue'].map({'N':'normal','T':'crc'})
# generate new cols
crc09_obs['Study'] = 'crc09'
crc09_obs['Glob_Sample'] = 'crc09-' + crc09_obs['Sample']
crc09_obs['Glob_Patient'] = 'crc09-' + crc09_obs['Patient']
crc09_obs['Glob_CellID'] = 'crc09-' + crc09_obs.index.astype(str)
crc09_obs = crc09_obs.set_index("Glob_CellID", drop=False)
crc09_obs['CellTypeFull'] = crc09_obs['clTopLevel'] + '_' + crc09_obs['CellType'] + '_' + crc09_obs['cl295v11SubFull']
adata_crc09.obs = crc09_obs
del crc09_obs
# only unsorted data is wanted, as we want to analysis cell proportion
adata_crc09 = adata_crc09[adata_crc09.obs['PROCESSING_TYPE'] == 'unsorted'].copy()
adata_crc09.obs = adata_crc09.obs.drop(columns = ['clTopLevel','cl295v11SubFull','PROCESSING_TYPE'])

/home/zhekai/miniconda3/envs/pm/lib/python3.13/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [13]:
# dealing var and index
adata_crc09.var["gene_ids"].is_unique
adata_crc09.var['gene_name'] = adata_crc09.var.index
adata_crc09.var = adata_crc09.var.rename(columns= {"gene_ids":"gene_id"})
# remove ADT(mainly membrane molecules involving in immunology)
adata_crc09 = adata_crc09[:, adata_crc09.var["gene_id"].str.startswith("ENSG")].copy()
# remove version suffix
# .str is the pd.Serie accessor to iterate through all elements
adata_crc09.var["gene_id"] = (adata_crc09.var["gene_id"].str.split(".").str[0])

/home/zhekai/miniconda3/envs/pm/lib/python3.13/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [14]:
adata_crc09.var["gene_id"].is_unique
len(adata_crc09.var_names[adata_crc09.var_names.duplicated()])
#####
# since ENSG ids are duplicated (lucky enough only 28 of them))
# we need to add them up. The following is written by ChatGPT
#####
# map duplicated gene_id -> one column
codes, unique_gene_ids = pd.factorize(
    adata_crc09.var["gene_id"],
    sort=False
)

mapping = sp.sparse.csr_matrix(
    (
        np.ones(adata_crc09.n_vars, dtype=np.int8),
        (np.arange(adata_crc09.n_vars), codes)
    ),
    shape=(adata_crc09.n_vars, len(unique_gene_ids))
)

# sum counts of duplicated genes
counts = adata_crc09.layers["counts"] @ mapping

# construct new var
var = (
    adata_crc09.var
    .groupby("gene_id", sort=False)
    .first()
    .loc[unique_gene_ids]
)

var["gene_id"] = var.index
var.index.name = None

# rebuild AnnData
adata_crc09 = ad.AnnData(
    X=None,
    obs=adata_crc09.obs.copy(),
    var=var,
)

adata_crc09.layers["counts"] = counts.astype(np.int32)

#####
# End of AI generation content
#####

adata_crc09.var = adata_crc09.var.set_index('gene_id',drop=False)
adata_crc09.var.index.name = None
adata_crc09.var = adata_crc09.var.drop(columns = ['genome','feature_types'])

In [15]:
adata_crc09.var["gene_id"].is_unique
len(adata_crc09.var_names[adata_crc09.var_names.duplicated()])

0

In [28]:
### ======
# crc10 data (scRNA-seq, SMC+KUL3)
### ======
# read in data
adata_crc10_KUL3 = sc.read_h5ad(DataRootDir / "crc10/lee-2020-KUL3.h5ad")
adata_crc10_SMC = sc.read_h5ad(DataRootDir / "crc10/lee-2020-SMC.h5ad")

for adata in [adata_crc10_KUL3,adata_crc10_SMC]:
    adata.layers['counts']= ut.to_int32_counts(adata.X.copy())
    del adata.X
del adata

In [29]:
# KUL3 obs
KUL3_obs = adata_crc10_KUL3.obs.copy()
rename_dict = {'Cell_type':'CellType','Class':'Tissue'}
KUL3_obs = KUL3_obs.rename(columns = rename_dict)
# transform categorical to string
for name in list(KUL3_obs.columns):
    KUL3_obs[name] = KUL3_obs[name].astype(str)
# generate new cols
KUL3_obs['Study'] = 'crc10-KUL3'
KUL3_obs['Glob_Sample'] = 'crc10' + KUL3_obs['Sample']
KUL3_obs['Glob_Patient'] = 'crc10-' + KUL3_obs['Patient']
KUL3_obs['Glob_CellID'] = 'crc10-' + KUL3_obs.index.astype(str)
KUL3_obs['CellID'] = KUL3_obs.index
KUL3_obs['CellTypeFull'] = KUL3_obs['CellType'] + '_' + KUL3_obs['Cell_subtype']
KUL3_obs['Batch'] = 'scRNA'
KUL3_obs = KUL3_obs.set_index("Glob_CellID", drop=False)
KUL3_obs = KUL3_obs.drop(columns = ['study','Cell_subtype'])
KUL3_obs['Tissue'] = KUL3_obs['Tissue'].map({'Tumor':'crc','Normal':'normal','Border':'crc'})
adata_crc10_KUL3.obs = KUL3_obs
# SMC obs
SMC_obs = adata_crc10_SMC.obs.copy()
rename_dict = {'Cell_type':'CellType','Class':'Tissue'}
SMC_obs = SMC_obs.rename(columns = rename_dict)
# transform categorical to string
for name in list(SMC_obs.columns):
    SMC_obs[name] = SMC_obs[name].astype(str)
# generate new cols
SMC_obs['Study'] = 'crc10-SMC'
SMC_obs['Glob_Sample'] = 'crc10' + SMC_obs['Sample']
SMC_obs['Glob_Patient'] = 'crc10-' + SMC_obs['Patient']
SMC_obs['Glob_CellID'] = 'crc10-' + SMC_obs.index.astype(str)
SMC_obs['CellID'] = SMC_obs.index
SMC_obs['CellTypeFull'] = SMC_obs['CellType'] + '_' + SMC_obs['Cell_subtype']
SMC_obs['Batch'] = 'scRNA'
SMC_obs = SMC_obs.set_index("Glob_CellID", drop=False)
SMC_obs = SMC_obs.drop(columns = ['Cell_subtype'])
SMC_obs['Tissue'] = SMC_obs['Tissue'].map({'Tumor':'crc','Normal':'normal','Border':'crc'})
adata_crc10_SMC.obs = SMC_obs

In [30]:
# concat
adata_crc10 = ad.concat([adata_crc10_KUL3,adata_crc10_SMC],
                        join="outer",merge="first",uns_merge=None,)
# var
adata_crc10.var = pd.DataFrame({'gene_name': adata_crc10.var_names})

/home/zhekai/miniconda3/envs/pm/lib/python3.13/functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [ ]:
print('Preview Data')
print('-----')
print(ut.preview_adata(adata_crc01))
print(ut.preview_adata(adata_crc03))
print(ut.preview_adata(adata_crc06))
print(ut.preview_adata(adata_crc09))
print(ut.preview_adata(adata_crc10))

In [ ]:
# BOOK MARK
# TODO: First do GTF -> ENSG ID transition

#### Step two: Data Integration

In [ ]:
# Merge crc01, crc03, crc06 by shared gene symbols
adata_crc01.var['gene_name'] = adata_crc01.var_names
adata_crc03.var['gene_name'] = adata_crc03.var_names
adata_crc06.var['gene_name'] = adata_crc06.var_names
adata_crc01.var['gene_id'] = ""
adata_crc03.var = adata_crc03.var.rename(columns={"gene_ids": "old_gene_id"})

# ── Gene filter: keep genes present in >=2 studies (var_names) ──
from collections import Counter
gene_counts = Counter()
gene_counts.update(adata_crc01.var_names)
gene_counts.update(adata_crc03.var_names)
gene_counts.update(adata_crc06.var_names)

keep_genes = sorted(g for g, c in gene_counts.items() if c >= 2)
print(f"Genes in >=2 studies: {len(keep_genes):,} / {len(gene_counts):,} total unique gene symbols")
print(f"  crc01: {adata_crc01.n_vars:,} genes")
print(f"  crc03: {adata_crc03.n_vars:,} genes")
print(f"  crc06: {adata_crc06.n_vars:,} genes")

# Pre-filter each AnnData to keep_genes, then outer-join concat
# join="outer" is safe here because the gene universe is already restricted to >=2 studies
adata_merged = ad.concat(
    [adata_crc03[:, [g for g in keep_genes if g in adata_crc03.var_names]],
     adata_crc06[:, [g for g in keep_genes if g in adata_crc06.var_names]],
     adata_crc01[:, [g for g in keep_genes if g in adata_crc01.var_names]]],
    join="outer",
    merge="first",
    uns_merge=None,
)

ut.preview_adata(adata_merged)
print(f"X dtype: {adata_merged.X.dtype}")
print(f"counts dtype: {adata_merged.layers['counts'].dtype}")

In [ ]:
adata_merged.var

#### Step three: unify vars
It is decided that gene symbol will be used as index, and ENSG id will be kept as back-ups
Format:
|gene_name|gene_id|
|-----|-----|
|gene A| ENSGXXXX|


Version: GRCh38.111


In [ ]:
# Map merged gene symbols to GRCh38.111 ENSG IDs
mapping_path = Path.home() / "t_rna/database/GRCh38.111/gene_id_gene_name.csv"
gene_mapping = pd.read_csv(mapping_path, dtype=str)
gene_mapping["gene_id"] = gene_mapping["gene_id"].str.replace(
    r"\.\d+$", "", regex=True
)

symbol_to_gene_ids = (
    gene_mapping.dropna(subset=["gene_name", "gene_id"])
    .groupby("gene_name")["gene_id"]
    .agg(lambda ids: ",".join(sorted(ids.unique())))
)

adata_merged.var["gene_id"] = adata_merged.var["gene_name"].map(
    symbol_to_gene_ids
)

mapped = adata_merged.var["gene_id"].notna()
multiple = adata_merged.var["gene_id"].str.contains(",", na=False)
adata_merged_unmapped_gene_names = adata_merged.var.index[~mapped].tolist()

print("adata_merged GRCh38.111 mapping")
print(f"mapped: {mapped.sum():,} / {adata_merged.n_vars:,} ({mapped.mean():.2%})")
print(f"cannot map to GRCh38.111: {(~mapped).sum():,}")
print(f"multiple ENSG IDs: {multiple.sum():,}")
print(
    "unmapped gene-name variable: "
    "adata_merged_unmapped_gene_names"
)

if "old_gene_id" in adata_merged.var.columns:
    old_ids = adata_merged.var["old_gene_id"].str.replace(
        r"\.\d+$", "", regex=True
    )
    new_ids = adata_merged.var["gene_id"]
    both_ids_present = old_ids.notna() & new_ids.notna()
    old_id_in_new_mapping = pd.Series(
        [
            pd.notna(old_id) and pd.notna(new_id) and old_id in new_id.split(",")
            for old_id, new_id in zip(old_ids, new_ids)
        ],
        index=adata_merged.var_names,
    )

    adata_merged_old_id_mismatch = adata_merged.var.loc[
        both_ids_present & ~old_id_in_new_mapping,
        ["gene_name", "old_gene_id", "gene_id"],
    ].copy()

    print("\nOld-to-GRCh38.111 comparison")
    print(f"both old and new IDs present: {both_ids_present.sum():,}")
    print(f"old IDs present in new mapping: {old_id_in_new_mapping.sum():,}")
    print(f"old/new ID mismatches: {len(adata_merged_old_id_mismatch):,}")
    print("mismatch table: adata_merged_old_id_mismatch")
else:
    print("\nOld-to-GRCh38.111 comparison: skipped (no old_gene_id column in adata_merged)")

# Add gene biotype as Class (GRCh38.111)
biotype_path = Path.home() / "t_rna/database/GRCh38.111/gene_id_gene_name_gene_biotype.csv"
biotype_mapping = pd.read_csv(biotype_path, dtype=str)
biotype_mapping["gene_id"] = biotype_mapping["gene_id"].str.replace(
    r"\.\d+$", "", regex=True
)

symbol_to_biotype = (
    biotype_mapping.groupby("gene_name")["gene_biotype"]
    .agg(lambda b: ",".join(sorted(b.unique())))
)

adata_merged.var["Class"] = adata_merged.var["gene_name"].map(
    symbol_to_biotype
)

mapped_class = adata_merged.var["Class"].notna()
multiple_class = adata_merged.var["Class"].str.contains(",", na=False)
print("\nClass (gene biotype) mapping")
print(f"Class mapped: {mapped_class.sum():,} / {adata_merged.n_vars:,} ({mapped_class.mean():.2%})")
print(f"multiple biotypes: {multiple_class.sum():,}")
print(adata_merged.var["Class"].value_counts().head(10))

In [ ]:
adata_merged.var

In [ ]:
# ── Biotype filter: remove non-informative gene classes (crc-atlas pattern) ──
# crc-atlas 20-Filter_mito.py:162-184 — rm_classes list
# Soft filter: gene is kept if ANY of its biotypes (for multi-biotype genes) is NOT in rm_classes
rm_classes = [
    "lncRNA", "processed_pseudogene", "unprocessed_pseudogene", "misc_RNA",
    "snRNA", "TEC", "transcribed_unprocessed_pseudogene", "snoRNA", "lincRNA",
    "transcribed_processed_pseudogene", "rRNA_pseudogene", "pseudogene", "antisense",
    "vault_RNA", "scRNA", "IG_pseudogene", "sRNA", "scaRNA", "artifact",
    "translated_processed_pseudogene", "unitary_pseudogene",
]

def _should_keep(class_val):
    if pd.isna(class_val):
        return False
    biotypes = str(class_val).split(",")
    return any(b.strip() not in rm_classes for b in biotypes)

keep_mask = adata_merged.var["Class"].apply(_should_keep)
print(f"Biotype filter: {keep_mask.sum():,} / {adata_merged.n_vars:,} genes kept")
print(f"Removed: {(~keep_mask).sum():,} genes")
if (~keep_mask).sum() > 0:
    removed_classes = adata_merged.var.loc[~keep_mask, "Class"].value_counts().head(10)
    print(f"Top removed classes:\n{removed_classes}")

adata_merged = adata_merged[:, keep_mask].copy()
print(f"After biotype filter: {adata_merged.shape}")

### Data Export

In [ ]:
export_dir = Path("../../result/curated/merged.h5ad")
adata_merged.write_h5ad(export_dir)